In [1]:
import os, io, sys, json, glob, time, hashlib, tempfile, subprocess, warnings, re, contextlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ============================================================
# Student / team settings — change only these
# ============================================================
TEAM_ID      = "team14"
STUDENT_ID   = "s1402"
COURSE       = "ITI113"
SEMESTER     = "26S1"
PROJECT_NAME = "airbnb-listings"
REGION       = "ap-southeast-1"

BUCKET = "nyp-26s1-iti113"
PREFIX = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

# --- The data layer: one project bucket, three prefixes (SSE + least-privilege IAM assumed)
RAW_PREFIX       = f"{PREFIX}/raw"          # immutable Listings.csv
PROCESSED_PREFIX = f"{PREFIX}/processed"    # versioned cleaned datasets
ARTIFACTS_PREFIX = f"{PREFIX}/artifacts"    # serialized pipeline, eval outputs, manifests

RANDOM_STATE = 42
TEST_SIZE    = 0.20
QUALITY_GATE_R2LOG = 0.65                   # same gate as Notebook 02

# --- Production object names
PIPELINE_NAME       = f"iti113-{TEAM_ID}-airbnb-price"
MODEL_PACKAGE_GROUP = f"iti113-{TEAM_ID}-airbnb-price-models"     # SageMaker Model Registry
REGISTRY_MODEL_NAME = f"iti113-{TEAM_ID}-airbnb-price-regressor"  # MLflow Model Registry
ENDPOINT_NAME       = f"iti113-{TEAM_ID}-airbnb-price-sls"
PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE   = "ml.m5.large"


# ============================================================
# Environment detection: SageMaker vs local
# ============================================================
try:
    import sagemaker, boto3
    session = sagemaker.Session()
    role = sagemaker.get_execution_role()
    LOCAL_MODE = False
    print(f"SageMaker mode | role: {role.split('/')[-1]}")
except Exception:
    LOCAL_MODE = True
    print("SageMaker not available -> LOCAL_MODE (scripts run as local subprocesses; "
          "SQLite MLflow store; local artifact folders mirror the S3 prefixes)")

# Local mirrors of the three prefixes
RAW_CANDIDATES  = ["/mnt/user-data/uploads/Listings.csv", "Listings.csv", "data/Listings.csv"]
RAW_LOCAL       = next((p for p in RAW_CANDIDATES if os.path.exists(p)), RAW_CANDIDATES[0])
LOCAL_PROCESSED = "pipeline_processed"      # this notebook's cleaned, versioned splits
LOCAL_ARTIFACTS = "artifacts"               # serialized pipeline + evaluation outputs
NB01_PROCESSED  = "processed"               # Notebook 01's outputs (consistency reference)
os.makedirs(LOCAL_ARTIFACTS, exist_ok=True)

# ============================================================
# MLflow App discovery (config saved by Notebook 01A)
# ============================================================
DEFAULT_MLFLOW_APP_ARN = "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PIGMOQJH46PS"
config_candidates = ([Path(f"mlflow_app_config_{TEAM_ID}.json"),
                      Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json")]
                     + [Path(p) for p in sorted(glob.glob(f"mlflow_app_config_{TEAM_ID}_*.json"))])
MLFLOW_APP_ARN, MLFLOW_CONFIG_SOURCE = None, None
for cand in config_candidates:
    if cand.exists():
        MLFLOW_APP_ARN = json.loads(cand.read_text()).get("MLFLOW_APP_ARN")
        MLFLOW_CONFIG_SOURCE = str(cand); break
if MLFLOW_APP_ARN is None and DEFAULT_MLFLOW_APP_ARN:
    MLFLOW_APP_ARN, MLFLOW_CONFIG_SOURCE = DEFAULT_MLFLOW_APP_ARN, "DEFAULT_MLFLOW_APP_ARN"

MLFLOW_EXPERIMENT = f"{COURSE}/{TEAM_ID}/Experiment1"
if LOCAL_MODE:
    MLFLOW_TRACKING_URI = "sqlite:///mlflow_local.db"      # Notebook 02's local store
    MLFLOW_CONFIG_SOURCE = MLFLOW_CONFIG_SOURCE or "local SQLite store"
else:
    if MLFLOW_APP_ARN is None:
        raise RuntimeError("No MLflow App config found. Run Notebook 01A or set DEFAULT_MLFLOW_APP_ARN.")
    MLFLOW_TRACKING_URI = MLFLOW_APP_ARN

# ============================================================
# Champion lineage: Notebook 02 hand-off
# ============================================================
# Local copy first; then the /artifacts zone, where Notebook 02 SS8A publishes it.
# The local file lives on ephemeral per-space disk -- S3 is what makes this
# hand-off survive a space reset, a teammate, or a CI runner.
if os.path.exists("best_model.json"):
    BEST = json.loads(Path("best_model.json").read_text())
    BEST_SOURCE = "local best_model.json"
elif not LOCAL_MODE:
    _key = f"{PREFIX}/artifacts/best_model.json"
    try:
        BEST = json.loads(boto3.client("s3", region_name=REGION)
                          .get_object(Bucket=BUCKET, Key=_key)["Body"].read())
        Path("best_model.json").write_text(json.dumps(BEST, indent=4))   # cache locally
        BEST_SOURCE = f"s3://{BUCKET}/{_key}"
    except Exception as _e:
        raise FileNotFoundError(
            f"best_model.json found neither locally nor at s3://{BUCKET}/{_key} ({_e}). "
            "Run Notebook 02 through SS8A, which publishes it.") from None
else:
    raise FileNotFoundError("best_model.json not found — run Notebook 02 first.")
print(f"Champion hand-off source: {BEST_SOURCE}")
print(f"\nChampion from Notebook 02: {BEST['best_run_name']}")
print(f"  run_id {BEST['best_run_id']} | test_r2_log {BEST['best_metrics']['test_r2_log']:.4f} "
      f"| gate passed: {BEST['quality_gate']['passed']}")
print(f"Raw data       : {RAW_LOCAL if LOCAL_MODE else f's3://{BUCKET}/{RAW_PREFIX}/Listings.csv'}")
print(f"MLflow config  : {MLFLOW_CONFIG_SOURCE}")
print(f"Tracking URI   : {MLFLOW_TRACKING_URI}")
print(f"Experiment     : {MLFLOW_EXPERIMENT}")
print(f"Pipeline       : {PIPELINE_NAME}")
print(f"Registry       : {REGISTRY_MODEL_NAME} (MLflow) / {MODEL_PACKAGE_GROUP} (SageMaker)")
print(f"Endpoint       : {ENDPOINT_NAME} (serverless)")
# WARNING: This deletes the entire model and all of its versions from the registry!
# client.delete_registered_model(name=REGISTRY_MODEL_NAME)
# print(f"Deleted the entire registered model: {REGISTRY_MODEL_NAME}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


SageMaker mode | role: SageMakerExecutionRole-ITI113-Team14
Champion hand-off source: local best_model.json

Champion from Notebook 02: team14_s1402_champion_refit
  run_id 3799806706b946aa88a316ffaa4eeb6d | test_r2_log 0.8708 | gate passed: True
Raw data       : s3://nyp-26s1-iti113/iti113/team14/data/airbnb-listings/raw/Listings.csv
MLflow config  : DEFAULT_MLFLOW_APP_ARN
Tracking URI   : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PIGMOQJH46PS
Experiment     : ITI113/team14/Experiment1
Pipeline       : iti113-team14-airbnb-price
Registry       : iti113-team14-airbnb-price-regressor (MLflow) / iti113-team14-airbnb-price-models (SageMaker)
Endpoint       : iti113-team14-airbnb-price-sls (serverless)


In [2]:
import mlflow
import mlflow.sklearn
import logging
logging.getLogger("mlflow").setLevel(logging.ERROR)
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

if not LOCAL_MODE:
    sm_client = boto3.client("sagemaker", region_name=REGION)
    tag_response = sm_client.list_tags(ResourceArn=MLFLOW_APP_ARN)
    mlflow_app_tags = {t["Key"]: t["Value"] for t in tag_response.get("Tags", [])}
    app_team_id = mlflow_app_tags.get("TeamId")
    if app_team_id != TEAM_ID:
        raise PermissionError(
            f"MLflow App TeamId tag mismatch. App TeamId={app_team_id}, notebook TEAM_ID={TEAM_ID}. "
            "Do not log to another team's MLflow App.")
    print(f"[OK] MLflow App tag TeamId={app_team_id} matches notebook TEAM_ID={TEAM_ID}")
else:
    print("[SKIP] LOCAL_MODE — no SageMaker App to tag-check; using the local SQLite store.")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

COMMON_TAGS = {
    "course": COURSE, "semester": SEMESTER, "team_id": TEAM_ID, "student_id": STUDENT_ID,
    "project_name": PROJECT_NAME, "task_type": "regression",
    "tracking_backend": "local_sqlite" if LOCAL_MODE else "sagemaker_mlflow_app",
    "notebook": "03_production_pipeline",
}

# --- Champion hyperparameters from the Notebook 02 hand-off (lineage, not copy-paste)
# Resolution order, most authoritative first:
#   1. best_model.json's `hyperparameters` block  (declared contract, NB02 §8B)
#   2. plain param names on the run               (champion refit)
#   3. `best_*` param names on the run            (legacy tuning-parent hand-off)
HP_KEYS = {"learning_rate": float, "max_leaf_nodes": int, "max_iter": int,
           "min_samples_leaf": int, "l2_regularization": float}

_champion_run = mlflow.get_run(BEST["best_run_id"])
_p = _champion_run.data.params

if isinstance(BEST.get("hyperparameters"), dict) and all(k in BEST["hyperparameters"] for k in HP_KEYS):
    _raw, _source = BEST["hyperparameters"], "best_model.json -> hyperparameters"
elif all(k in _p for k in HP_KEYS):
    _raw, _source = _p, f"run {BEST['best_run_id'][:12]}… params (champion refit)"
elif all(f"best_{k}" in _p for k in HP_KEYS):
    _raw = {k: _p[f"best_{k}"] for k in HP_KEYS}
    _source = f"run {BEST['best_run_id'][:12]}… params (legacy best_* prefix)"
else:
    raise KeyError(
        "Could not resolve champion hyperparameters.\n"
        f"  best_model.json keys : {sorted(BEST)}\n"
        f"  run parameter keys   : {sorted(_p)}\n"
        f"Expected a `hyperparameters` block in best_model.json, or {sorted(HP_KEYS)} "
        "on the run (bare or `best_`-prefixed). Re-run Notebook 02 §8B.")

# Exactly the five knobs train.py accepts. The refit run also carries early_stopping
# and random_state, which build_model_pipeline() sets itself — passing them through
# would collide and would leak into cell 26's hp_* params.
CHAMPION_PARAMS = {k: cast(_raw[k]) for k, cast in HP_KEYS.items()}

# pipeline_lib.build_model_pipeline() hardcodes HistGradientBoostingRegressor, and
# these five knobs are meaningless for anything else. Fail here rather than train an
# HGB using another family's settings.
_family = BEST.get("model_type", "HistGradientBoostingRegressor")
if _family != "HistGradientBoostingRegressor":
    raise RuntimeError(
        f"Notebook 02 selected {_family}, but pipeline_lib.build_model_pipeline() "
        "hardcodes HistGradientBoostingRegressor. Update and re-version pipeline_lib first.")

print(f"MLflow {mlflow.__version__} connected | experiment: {MLFLOW_EXPERIMENT}")
print(f"Champion hyperparameters (resolved from: {_source}):")
for k, v in CHAMPION_PARAMS.items():
    print(f"  {k:18s} = {v}")

_sel = BEST.get("selection")
if _sel:
    print(f"\nSelected in Notebook 02 as '{_sel['selected_run_name']}'")
    print(f"  on {_sel['metric']} = {_sel['value']:.4f}, "
          f"margin over runner-up {_sel.get('runner_up_margin', float('nan')):.4f}")
_proto = BEST.get("validation_protocol")
if _proto:
    _reads = _proto.get("test_decisions", _proto.get("test_set_reads", "n/a"))
    print(f"  protocol: fit {_proto['n_fit']:,} / val {_proto['n_val']:,} / "
          f"test {_proto['n_test']:,}; champion refit on {_proto['n_train_refit']:,}; "
          f"test decisions = {_reads}")

[OK] MLflow App tag TeamId=team14 matches notebook TEAM_ID=team14


MLflow 3.15.1 connected | experiment: ITI113/team14/Experiment1
Champion hyperparameters (resolved from: best_model.json -> hyperparameters):
  learning_rate      = 0.1
  max_leaf_nodes     = 127
  max_iter           = 300
  min_samples_leaf   = 50
  l2_regularization  = 1.0

Selected in Notebook 02 as 'team14_s1402_hgb_tuning_parent'
  on val_mae_log = 0.3262, margin over runner-up 0.0148
  protocol: fit 178,715 / val 44,679 / test 55,849; champion refit on 223,394; test decisions = 1


In [3]:
# ================================================================
# 0B. Presigned MLflow App UI links
# ----------------------------------------------------------------
# MLflow prints generic links like https://mlflow.sagemaker.<region>.app.aws/#/...
# Those are NOT presigned and will show a permission/session error. Generate a
# fresh presigned URL instead and append the UI route as a fragment.
#
# SECURITY: a presigned URL is a CREDENTIAL. Anyone holding it can act as this
# execution role inside the MLflow App until it expires. Do not paste one into
# the report, a screenshot, Slack, or a git commit. Cite the ARN plus the
# experiment/run IDs instead -- those are safe identifiers -- and regenerate a
# live link at demo time.
# ================================================================
if LOCAL_MODE:
    print("[SKIP] LOCAL_MODE — tracking URI is sqlite:///mlflow_local.db, which has no "
          "MLflow App UI. Presigned links apply only to the SageMaker MLflow App.")
else:
    import botocore

    # ExpiresInSeconds  = how long this link stays redeemable (short by design)
    # SessionExpiration = how long the resulting UI session lasts once opened
    MLFLOW_URL_EXPIRES_IN = 300
    MLFLOW_SESSION_DURATION = 36000

    def create_mlflow_app_presigned_url(fragment: str = "") -> str:
        """Return a presigned MLflow App URL, optionally deep-linked to a UI route."""
        kwargs = dict(Arn=MLFLOW_APP_ARN,
                      ExpiresInSeconds=MLFLOW_URL_EXPIRES_IN,
                      SessionExpirationDurationInSeconds=MLFLOW_SESSION_DURATION)
        try:
            resp = sm_client.create_presigned_mlflow_app_url(**kwargs)
        except botocore.exceptions.ParamValidationError:
            # Older botocore may not accept the expiry parameters — fall back to defaults.
            resp = sm_client.create_presigned_mlflow_app_url(Arn=MLFLOW_APP_ARN)

        base = resp.get("AuthorizedUrl") or resp.get("Url")
        if not base:
            raise RuntimeError(f"No AuthorizedUrl/Url in response: {resp}")

        base = base.split("#", 1)[0]          # drop any fragment before adding ours
        return base + "#" + fragment.lstrip("#") if fragment else base

    def print_mlflow_presigned_links(experiment_id=None, run_id=None, label=""):
        """Print presigned links for an experiment and (optionally) a specific run."""
        if label:
            print(f"--- {label} ---")
        if experiment_id is None:
            print(create_mlflow_app_presigned_url())
            return
        print("Experiment:")
        print("  " + create_mlflow_app_presigned_url(f"/experiments/{experiment_id}"))
        if run_id:
            print("Run:")
            print("  " + create_mlflow_app_presigned_url(
                f"/experiments/{experiment_id}/runs/{run_id}"))
        print(f"(redeemable for {MLFLOW_URL_EXPIRES_IN}s; "
              f"session lasts {MLFLOW_SESSION_DURATION}s — regenerate when stale)")

    # Resolve the experiment id for the experiment this notebook is writing to
    _exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
    MLFLOW_EXPERIMENT_ID = _exp.experiment_id if _exp else None

    print(f"MLflow App ARN : {MLFLOW_APP_ARN}")
    print(f"Experiment     : {MLFLOW_EXPERIMENT} (id {MLFLOW_EXPERIMENT_ID})")
    print(f"Champion run   : {BEST['best_run_id']}\n")
    print_mlflow_presigned_links(MLFLOW_EXPERIMENT_ID, BEST["best_run_id"],
                                label="Notebook 02 champion")


MLflow App ARN : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-PIGMOQJH46PS
Experiment     : ITI113/team14/Experiment1 (id 1)
Champion run   : 3799806706b946aa88a316ffaa4eeb6d

--- Notebook 02 champion ---
Experiment:


  https://app-PIGMOQJH46PS.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IjdVMk1OViIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNDVEZUhNVXpmQVhpREY4RjRWaFhsZ1BpMTg4OU5wNlpkU1BUUUdpSjhMVmdBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGNlRrZE5aVFJ5TlRCR1F6UlFTV3RhYjNkT01UQXhWV0ZyTlc1cFpEaEtPRGxoY1VaMmRHZzRWR3A2YnpWa1ZVcE1SVEJrWm1wT1dHcHBkMUZUZUdkQmR6MDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFXOU9rNFNrNjNqejRieGIwVUE0TGJJQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF3ZDJVeVRqZUVDTTZDR3BXOENBUkNBTzZneWVxZ1Y0ZWtKUGdybzJsVGhQQ0VpRjQranNEb204MTdyQ2VraCt6b08xM3cyMXZBajI1Z1huRHJmQkxGQW5IU2krU0V2ZURFSnlhZWVBZ0FBRUFCdVdsRVUyNkJUY0VwWGYrQm5FMGVEa2tHT3RhM3VOb2F6VmRwWExWcVp1UjZpcXVvOUt6bVdjZjZmVmV2bDdudi8vLy8vQUFBQUFRQ

  https://app-PIGMOQJH46PS.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IlIyQ0JOUCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNFBTQWsrK3lNSHo0cDdzY2RjQklDWExkZy9XU0x3YXQ4QnNIYUo2Q05qYkVBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGb1pFMVBSV2RKZWxaWFpFbG1ObTR2VTJSNVMyRXdRMk4wTWxJNWVFUmlkazA0VTBGNFMxQmpRbmR1VEVKRlZFdDJTbGhEYm1OV0syTm9kamxqZUZaV1VUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFhcER1a1JGNWNJbGJpVWtqOHNFSWp3QUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF4VkEvREhiaGtsSFo4Rk9Mc0NBUkNBTzlsSUszZ1loVjFZSTh3VlN0OWJKWXhHejZQOWtTcVBqdGlvWWhmeEVvbWtKQWNRTEN4SUgrbWY2d2ZIWXhwblNIYlJyL21DT1BXcmwrN3RBZ0FBRUFENm1panJRZzlmQms1UjduQ3VDb3lPaGx1cFAvSzNsQWdrRkVJSkZ6WFlTU1RFdkoxci9oUVd3UVNCWFJUNkFYYi8vLy8vQUFBQUFRQ